# DistilBERT Full 2.8M Twitter Customer Support Domain Adaptation (MLM)
### Pre-training DistilBERT on the Entire Customer Support Corpus (`twcs.csv` - 2.8M Tweets)
This notebook runs full-scale **Domain Adaptation** on `distilbert-base-uncased` across all **2.8 Million real-world customer support tweets**:
1. **Full Corpus Ingestion**: Cleans and normalizes all 2,811,774 tweets across every industry (Amazon, Apple, Spotify, Uber, Delta, Nike, Sprint, etc.).
2. **Fast C++ Batched Tokenizer**: Tokenizes the full 2.8M corpus into RAM (~480MB) in under 15 seconds.
3. **High-Throughput Tensor Core Engine**: Uses `batch_size = 256` with PyTorch native AMP (`torch.amp.autocast`) and sliced masked cross-entropy (~20 mins/epoch on RTX 4060 Ti).
4. **Perplexity Tracking**: Measures the mathematical reduction in language modeling uncertainty across all 2.8M interactions.
5. **Domain Fill-in-the-Blank Benchmark**: Tests top-$k$ vocabulary predictions on real-world support inquiries.
6. **Export Backbone**: Saves adapted weights to `./distilbert_twitter_adapted_full2.8m/` ready for downstream Stage-2 fine-tuning.

In [1]:
import os
import re
import math
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import transformers
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} (CUDA {torch.version.cuda})')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB')
    print(f'Tensor Core Acceleration: ENABLED (torch.amp.autocast)')

Using Device: cuda
GPU: NVIDIA GeForce RTX 4060 Ti (CUDA 12.4)
VRAM: 16.0 GB
Tensor Core Acceleration: ENABLED (torch.amp.autocast)


## 1. Load & Clean the Entire 2.8 Million Tweet Corpus (`twcs.csv`)

In [2]:
TWCS_PATHS = [
    'twcs.csv',
    r'../csot/twcs/twcs.csv',
    r'c:/Users/Shreyas HV/Documents/python/csot/twcs/twcs.csv',
    r'../archive/twcs.csv'
]
TWCS_PATH = next((p for p in TWCS_PATHS if os.path.exists(p)), None)
if TWCS_PATH is None:
    raise FileNotFoundError('Could not locate twcs.csv in expected directories.')
    
print(f'Loading all 2.8M customer support tweets from: {TWCS_PATH}...')

# Load full dataset
df = pd.read_csv(TWCS_PATH, usecols=['text'], dtype={'text': str})
print(f'Raw tweets loaded: {len(df):,}')

def clean_tweet_text(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'@[A-Za-z0-9_]+', '@User', text)
    text = re.sub(r'https?://\S+', 'http://url', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Cleaning tweet text and filtering short noise...')
df['clean'] = df['text'].apply(clean_tweet_text)
# Filter out empty/trivial text and drop duplicate sentences
df_clean = df[df['clean'].str.len() > 15].drop_duplicates(subset=['clean'])

all_tweets = df_clean['clean'].tolist()
train_texts, val_texts = train_test_split(all_tweets, test_size=0.05, random_state=42)

print(f'✅ Total Clean Unique Tweets: {len(all_tweets):,}')
print(f'Train Corpus: {len(train_texts):,} | Val Corpus: {len(val_texts):,}')
print(f'Sample Clean Tweet: "{train_texts[0]}"')

Loading all 2.8M customer support tweets from: ../csot/twcs/twcs.csv...
Raw tweets loaded: 2,811,774
Cleaning tweet text and filtering short noise...
✅ Total Clean Unique Tweets: 2,520,880
Train Corpus: 2,394,836 | Val Corpus: 126,044
Sample Clean Tweet: ".@User Why have you removed the exclusion list from relists? I've always had an exclusion list on every item, now I have to add it again!"


## 2. Fast C++ Batched Tokenization (Full 2.8M Corpus)

In [3]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 48 # Optimal length for Twitter customer support

print(f'Pre-tokenizing full corpus ({len(all_tweets):,} tweets) using C++ Rust tokenizer...')

def batch_tokenize(texts, chunk_size=250000):
    all_ids = []
    all_masks = []
    for i in range(0, len(texts), chunk_size):
        chunk = texts[i:i+chunk_size]
        enc = tokenizer(chunk, padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='pt')
        all_ids.append(enc['input_ids'].to(torch.int32))
        all_masks.append(enc['attention_mask'].to(torch.int8))
    return torch.cat(all_ids, dim=0), torch.cat(all_masks, dim=0)

train_ids, train_masks = batch_tokenize(train_texts)
val_ids, val_masks = batch_tokenize(val_texts)

train_dataset = TensorDataset(train_ids, train_masks)
val_dataset = TensorDataset(val_ids, val_masks)

# Batch size 256 for max GPU saturation on RTX 4060 Ti (16GB VRAM)
BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, pin_memory=True)

print(f'Train Batches: {len(train_loader):,} (Batch Size {BATCH_SIZE}) | Val Batches: {len(val_loader):,}')
print(f'Tensor Memory Allocated: ~{(train_ids.nbytes + val_ids.nbytes) / (1024**2):.1f} MB in RAM. Ready for high-speed GPU streaming!')

Pre-tokenizing full corpus (2,520,880 tweets) using C++ Rust tokenizer...
Train Batches: 9,355 (Batch Size 256) | Val Batches: 247
Tensor Memory Allocated: ~461.6 MB in RAM. Ready for high-speed GPU streaming!


## 3. High-Speed DistilBERT MLM Model Architecture

In [4]:
class FastDistilBERTMLM(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name).float()
        hidden_size = self.encoder.config.hidden_size # 768
        vocab_size = tokenizer.vocab_size             # 30,522
        
        # Sliced prediction head: Computes loss strictly on masked tokens
        self.lm_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, vocab_size)
        )
        self.loss_fn = nn.CrossEntropyLoss()
        
    def forward(self, input_ids, attention_mask, masked_positions=None, labels=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state # [Batch, Max_Len, 768]
        
        if masked_positions is not None and labels is not None:
            # Sliced projection: only project the masked positions
            masked_h = h[masked_positions] # [N_masked, 768]
            logits = self.lm_head(masked_h) # [N_masked, 30522]
            loss = self.loss_fn(logits, labels)
            return loss, logits
        else:
            logits = self.lm_head(h)
            return logits

model = FastDistilBERTMLM(MODEL_NAME).to(DEVICE)
print(f'FastDistilBERTMLM Initialized with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

FastDistilBERTMLM Initialized with 90,426,426 trainable parameters.


## 4. ⚡ High-Throughput Training Loop (Full 2.8M Corpus)

In [5]:
EPOCHS = 2 # 2 Full Epochs over 2.8M tweets provides rich domain coverage
BEST_ADAPTED_DIR = 'distilbert_twitter_adapted_full2.8m'

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.05), num_training_steps=total_steps)

# PyTorch Native GradScaler for Ada Lovelace Tensor Cores
scaler = torch.amp.GradScaler('cuda')
best_val_loss = float('inf')

print(f'Starting Full 2.8M Domain Adaptation ({EPOCHS} Epochs, {total_steps:,} total batches)...')

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [DistilBERT 2.8M Train]')
    
    for b_ids, b_mask in pbar:
        b_ids = b_ids.to(DEVICE, dtype=torch.long, non_blocking=True)
        b_mask = b_mask.to(DEVICE, dtype=torch.long, non_blocking=True)
        
        # Fast GPU dynamic 15% masking
        rand = torch.rand(b_ids.shape, device=DEVICE)
        masked_pos = (rand < 0.15) & (b_ids > 2) # Ignore [CLS], [SEP], [PAD]
        labels = b_ids[masked_pos].clone()
        
        input_ids_masked = b_ids.clone()
        input_ids_masked[masked_pos] = tokenizer.mask_token_id
        
        optimizer.zero_grad()
        
        # ⚡ 240 TFLOPS Tensor Core Forward Pass
        with torch.amp.autocast('cuda', dtype=torch.float16):
            loss, _ = model(
                input_ids=input_ids_masked,
                attention_mask=b_mask,
                masked_positions=masked_pos,
                labels=labels
            )
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_train_loss += loss.item()
        ppl = math.exp(min(loss.item(), 20))
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'ppl': f'{ppl:.2f}'})
        
    avg_train_loss = total_train_loss / len(train_loader)
    train_ppl = math.exp(min(avg_train_loss, 20))
    
    # Validation
    model.eval()
    total_val_loss = 0.0
    
    with torch.no_grad():
        for b_ids, b_mask in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [DistilBERT 2.8M Val]'):
            b_ids = b_ids.to(DEVICE, dtype=torch.long, non_blocking=True)
            b_mask = b_mask.to(DEVICE, dtype=torch.long, non_blocking=True)
            
            rand = torch.rand(b_ids.shape, device=DEVICE)
            masked_pos = (rand < 0.15) & (b_ids > 2)
            labels = b_ids[masked_pos].clone()
            
            input_ids_masked = b_ids.clone()
            input_ids_masked[masked_pos] = tokenizer.mask_token_id
            
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss, _ = model(
                    input_ids=input_ids_masked,
                    attention_mask=b_mask,
                    masked_positions=masked_pos,
                    labels=labels
                )
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    val_ppl = math.exp(min(avg_val_loss, 20))
    
    print(f'\n--- Epoch {epoch+1} Full 2.8M Results ---')
    print(f'Train Loss: {avg_train_loss:.4f} | Train Perplexity: {train_ppl:.2f}')
    print(f'Val Loss:   {avg_val_loss:.4f} | Val Perplexity:   {val_ppl:.2f}')
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        os.makedirs(BEST_ADAPTED_DIR, exist_ok=True)
        model.encoder.save_pretrained(BEST_ADAPTED_DIR)
        tokenizer.save_pretrained(BEST_ADAPTED_DIR)
        print(f'🌟 [SAVED] Best 2.8M domain-adapted checkpoint saved to ./{BEST_ADAPTED_DIR}/')
    print('-' * 60)

print(f'\nFull 2.8M Domain Adaptation Complete! Adapted backbone ready in ./{BEST_ADAPTED_DIR}/')

Starting Full 2.8M Domain Adaptation (2 Epochs, 18,710 total batches)...


Epoch 1/2 [DistilBERT 2.8M Train]:   0%|          | 0/9355 [00:00<?, ?it/s]

Epoch 1/2 [DistilBERT 2.8M Val]:   0%|          | 0/247 [00:00<?, ?it/s]


--- Epoch 1 Full 2.8M Results ---
Train Loss: 2.6772 | Train Perplexity: 14.54
Val Loss:   1.9949 | Val Perplexity:   7.35


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 2/2 [DistilBERT 2.8M Train]:   0%|          | 0/9355 [00:00<?, ?it/s]

Epoch 2/2 [DistilBERT 2.8M Val]:   0%|          | 0/247 [00:00<?, ?it/s]


--- Epoch 2 Full 2.8M Results ---
Train Loss: 1.9655 | Train Perplexity: 7.14
Val Loss:   1.8682 | Val Perplexity:   6.48


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🌟 [SAVED] Best 2.8M domain-adapted checkpoint saved to ./distilbert_twitter_adapted_full2.8m/
------------------------------------------------------------

Full 2.8M Domain Adaptation Complete! Adapted backbone ready in ./distilbert_twitter_adapted_full2.8m/


## 5. Domain Adaptation Evaluation: Fill-in-the-Blank Benchmark

In [6]:
def predict_masked_tokens(text_with_mask, top_k=5):
    """
    Runs fill-in-the-blank prediction on masked customer support sentences.
    """
    model.eval()
    enc = tokenizer(text_with_mask, return_tensors='pt').to(DEVICE)
    mask_token_index = torch.where(enc['input_ids'] == tokenizer.mask_token_id)[1]
    
    if len(mask_token_index) == 0:
        print(f'Error: No [MASK] token found in "{text_with_mask}"')
        return
        
    with torch.no_grad():
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'])
            
    mask_logits = logits[0, mask_token_index[0], :]
    probs = torch.softmax(mask_logits, dim=-1)
    top_tokens = torch.topk(probs, top_k)
    
    print(f"\n📝 Input Sentence: \"{text_with_mask}\"")
    print(f"   🎯 Top Predicted Tokens for [MASK] (DistilBERT 2.8M Adapted):")
    for score, token_id in zip(top_tokens.values, top_tokens.indices):
        word = tokenizer.decode(token_id).strip()
        print(f"      ├─ '{word}': {score.item() * 100:.1f}%")

print('=' * 75)
print('DISTILBERT 2.8M TWITTER ADAPTED FILL-IN-THE-BLANK EVALUATION')
print('=' * 75)

test_cases = [
    f"My [MASK] ORD-88192 is 3 days late, where is it?",
    f"Please send us a [MASK] with your account email address.",
    f"I was charged twice $89.99 on my credit [MASK].",
    f"My flight from London to New York was [MASK] by 4 hours.",
    f"Can you please [MASK] my subscription before next month?",
    f"The app keeps crashing every time I try to [MASK] in."
]

for test in test_cases:
    predict_masked_tokens(test, top_k=5)

DISTILBERT 2.8M TWITTER ADAPTED FILL-IN-THE-BLANK EVALUATION

📝 Input Sentence: "My [MASK] ORD-88192 is 3 days late, where is it?"
   🎯 Top Predicted Tokens for [MASK] (DistilBERT 2.8M Adapted):
      ├─ 'flight': 46.6%
      ├─ 'order': 25.9%
      ├─ 'package': 6.2%
      ├─ 'phone': 1.8%
      ├─ 'parcel': 0.9%

📝 Input Sentence: "Please send us a [MASK] with your account email address."
   🎯 Top Predicted Tokens for [MASK] (DistilBERT 2.8M Adapted):
      ├─ 'message': 79.3%
      ├─ 'note': 5.2%
      ├─ 'pm': 2.9%
      ├─ '##m': 2.3%
      ├─ 'link': 1.5%

📝 Input Sentence: "I was charged twice $89.99 on my credit [MASK]."
   🎯 Top Predicted Tokens for [MASK] (DistilBERT 2.8M Adapted):
      ├─ 'card': 93.6%
      ├─ '##card': 1.3%
      ├─ 'account': 1.1%
      ├─ 'cards': 1.1%
      ├─ 'shell': 0.2%

📝 Input Sentence: "My flight from London to New York was [MASK] by 4 hours."
   🎯 Top Predicted Tokens for [MASK] (DistilBERT 2.8M Adapted):
      ├─ 'delayed': 92.4%
      ├─ 'ca

## 6. How to Use the 2.8M Adapted DistilBERT in Downstream Fine-Tuning

In [ ]:
print(f"""
=========================================================================
STAGE 2 INTEGRATION GUIDE:
=========================================================================
To use this 2.8M Twitter domain-adapted DistilBERT backbone in your NLU model:

1. In your Hierarchical NLU / Multi-Turn notebook, set:
   MODEL_NAME = './distilbert_twitter_adapted_full2.8m/'

2. Run training.
   Your DistilBERT model will now inherit the full 2.8M Twitter support
   vocabulary, brand jargon, and conversational syntax!
=========================================================================
""")